# 청주시 공공자전거 분석 — 전처리 (최종본, 2026.10.7)

**목표:** 원본 데이터를 모델에 바로 넣을 수 있는 두 개의 표로 만든다. 시각화는 별도 단계에서 한다.

| 단계 | 내용 | 결과 파일 (`02_전처리/`) |
|---|---|---|
| 0 | Drive 연결, 파일 자동 정리 | – |
| 1 | 타슈 대여이력 → 대여소별 일평균 대여건수 (2025.4~2026.3) | `타슈_대여소별수요.csv` |
| 2 | 버스정류장 → 대전·청주 추출 | `버스정류장_대전.csv`, `버스정류장_청주.csv` |
| 3 | 행정동 경계 (대전 82·청주 43) | `행정동목록_대전/청주.csv` |
| 4 | 산단·대학·철도역 좌표 검증 | `시설좌표_최종.csv` |
| 5 | 행정동별 종사자 수 | `행정동별_종사자수.csv` |
| 6 | 행정동별 인구 | `행정동별_인구.csv` |
| 7 | **대전 학습 표** (대여소 × 변수 7개 + 수요) | **`학습데이터_대전.csv`** |
| 8 | **청주 예측 표** (500m 격자 × 변수 7개) | **`예측데이터_청주격자.csv`** |

**사용법:** Drive `데이터` 폴더에 원본 파일을 모두 넣고, 위에서부터 한 셀씩 실행한다(Shift+Enter).

## 0. 준비: Drive 연결과 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ▼ Drive 안의 프로젝트 폴더 경로 (폴더 이름을 다르게 만들었다면 여기만 고친다)
BASE = '/content/drive/MyDrive/데이터'

In [ ]:
# ▼ '데이터' 폴더에 섞여 있는 파일을 종류별 하위 폴더로 정리한다 (여러 번 실행해도 안전)
import os, shutil, unicodedata

RULES = [
    (['타슈', 'tashu', '대여이력'],          '01_원본/tashu'),
    (['정류장', '버스', 'bus', 'BUS'],       '01_원본/bus'),
    (['종사자', '사업체', '산업분류'],        '01_원본/workers'),
    (['인구', '주민', '세대'],               '01_원본/population'),
    (['시설', 'facilities'],                 '01_원본'),
    (['자전거도로'],                          '01_원본/bikeroad'),
]
assert os.path.isdir(BASE), f'폴더를 찾을 수 없습니다: {BASE} (왼쪽 폴더 아이콘에서 경로를 복사해 BASE를 고치세요)'
for f in sorted(os.listdir(BASE)):
    src = os.path.join(BASE, f)
    if os.path.isdir(src):
        continue                      # 이미 만들어진 폴더는 건너뜀
    # Drive는 한글 파일 이름을 자모가 분리된 형태(NFD)로 줄 때가 있어 비교 전에 하나로 합친다(NFC)
    fn = unicodedata.normalize('NFC', f)
    dest = next((d for keys, d in RULES if any(k in fn for k in keys)), None)
    if dest is None:
        print(f'❓ 분류 못 함 → {fn}')
        continue
    os.makedirs(os.path.join(BASE, dest), exist_ok=True)
    name = '시설좌표.csv' if dest == '01_원본' else fn
    shutil.move(src, os.path.join(BASE, dest, name))
    print(f'✅ {fn}  →  {dest}')
print('정리 완료')

In [ ]:
import os, glob
import numpy as np
import pandas as pd

RAW  = os.path.join(BASE, '01_원본')
PRE  = os.path.join(BASE, '02_전처리')
OUT  = os.path.join(BASE, '03_결과')
for d in [PRE, OUT]:
    os.makedirs(d, exist_ok=True)

def read_csv_any(path, **kw):
    """한글 CSV를 인코딩을 바꿔가며 읽는다."""
    for enc in ['cp949', 'utf-8-sig', 'utf-8', 'euc-kr']:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False, **kw)
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise ValueError(f'인코딩을 알 수 없습니다: {path}')

def find_col(cols, include, exclude=(), any_of=()):
    """컬럼 이름에 특정 글자가 들어간 컬럼을 찾는다."""
    for c in cols:
        name = str(c).replace(' ', '')
        if all(k in name for k in include) and not any(k in name for k in exclude) \
           and (not any_of or any(k in name for k in any_of)):
            return c
    return None

print('경로 확인:', BASE)
print('타슈 파일:', glob.glob(os.path.join(RAW, 'tashu', '*.csv')) or glob.glob(os.path.join(RAW, 'tashu', '*.zip')), '(zip은 1-1 셀에서 자동으로 풀림)')
print('버스 파일:', glob.glob(os.path.join(RAW, 'bus', '*.csv')))

# 지도 배경: API 키 없이 쓸 수 있는 Esri 회색 지도 (OpenStreetMap 공식 서버는 Colab 요청을 차단함)
TILE_URL  = 'https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}'
TILE_ATTR = 'Esri, HERE, Garmin, OpenStreetMap contributors'

## 1. 타슈 대여이력 → 대여소별 수요

### 1-1. 컬럼 확인
파일이 크므로 먼저 앞부분만 읽어 컬럼 이름을 확인하고, 필요한 컬럼을 자동으로 찾는다.

In [ ]:
# zip 파일이 있으면 자동으로 풀어서 CSV만 tashu 폴더에 꺼낸다 (zip 안에 폴더가 있어도 OK)
import zipfile, shutil
TASHU_DIR = os.path.join(RAW, 'tashu')
for zp in glob.glob(os.path.join(TASHU_DIR, '*.zip')):
    with zipfile.ZipFile(zp) as z:
        for i, info in enumerate(z.infolist()):
            if info.filename.lower().endswith('.csv'):
                target = os.path.join(TASHU_DIR, f'타슈_월별_{i:02d}.csv')
                old = os.path.join(TASHU_DIR, f'tashu_{i:02d}.csv')   # 예전 이름으로 풀린 파일은 이름만 바꿈
                if os.path.exists(old) and not os.path.exists(target):
                    os.rename(old, target)
                    print('이름 변경:', os.path.basename(target))
                if not os.path.exists(target):
                    with z.open(info) as src, open(target, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
                    print('압축 해제:', target)

tashu_files = sorted(glob.glob(os.path.join(TASHU_DIR, '*.csv')))
print('타슈 CSV:', [os.path.basename(f) for f in tashu_files])
assert tashu_files, '타슈 CSV가 없습니다. Drive의 01_원본/tashu/ 폴더에 zip 또는 csv를 넣었는지 확인하세요.'
sample = read_csv_any(tashu_files[0], nrows=1000)
print('전체 컬럼:', list(sample.columns))
display(sample.head())

cols = sample.columns
auto = {
    'station': find_col(cols, ['대여소'], exclude=['반납', '명', '주소', '위도', '경도']),
    'lat':     find_col(cols, ['위도'], exclude=['반납']) or find_col(cols, ['Y좌표'], exclude=['반납']),   # Y좌표 = 위도
    'lon':     find_col(cols, ['경도'], exclude=['반납']) or find_col(cols, ['X좌표'], exclude=['반납']),   # X좌표 = 경도
    'date':    find_col(cols, ['대여'], exclude=['반납', '대여소', '이용'], any_of=['일시', '일자', '날짜']),
    'dist':    find_col(cols, ['거리']),
}
print('\n자동으로 찾은 컬럼:')
for k, v in auto.items():
    print(f'  {k:8s} → {v}')

### 1-2. 사용할 컬럼 지정
타슈 실제 파일 기준으로 컬럼 이름을 고정했다. 좌표는 `대여_Y좌표`가 위도, `대여_X좌표`가 경도이고, 이용거리 단위는 km이다.

In [ ]:
# ▼ 사용할 컬럼 지정 (타슈 실제 컬럼 이름 기준)
COL_STATION = '대여_대여소ID'   # 대여소 번호
COL_LAT     = '대여_Y좌표'      # 위도 (Y = 위도)
COL_LON     = '대여_X좌표'      # 경도 (X = 경도)
COL_DATE    = '대여일시'        # 대여 날짜·시간
COL_DIST    = '이용거리(km)'    # 이용 거리(km)

# 컬럼 이름이 실제 파일에 있는지 확인
need = [COL_STATION, COL_LAT, COL_LON, COL_DATE, COL_DIST]
missing = [c for c in need if c not in sample.columns]
assert not missing, f'파일에 없는 컬럼: {missing} → 1-1 셀의 전체 컬럼 목록과 철자를 비교하세요.'

# 좌표 값이 위경도(도 단위)인지 미리 확인
print(sample[[COL_LAT, COL_LON]].head())
print('사용할 컬럼:', need)

### 1-3. 분석 기간만 읽기 → 대여소별 집계
**분석 기간: 최근 1년(2025.4 ~ 2026.3).** 사계절이 한 번씩 모두 들어가 계절 편향이 없고, 타슈 대여소가 최근 빠르게 늘어난 만큼 가장 최근 배치 상태를 반영한다. 파일을 하나씩 읽으면서 기간 밖 기록은 바로 버려 메모리를 아낀다.

**일평균 계산:** 기간 중에 새로 생긴 대여소는 운영 일수가 짧으므로, 전체 기간이 아니라 **대여소별 실제 운영 일수**(첫 대여일~마지막 대여일)로 나눈다. 운영 30일 미만 대여소는 값이 불안정해 제외한다.

In [ ]:
# ▼ 분석 기간 (회의 결정에 따라 바꿀 수 있음)
PERIOD_START = '2025-04-01'
PERIOD_END   = '2026-03-31'
MIN_ACTIVE_DAYS = 30

use = [c for c in [COL_STATION, COL_LAT, COL_LON, COL_DATE, COL_DIST] if c]
p_start, p_end = pd.Timestamp(PERIOD_START), pd.Timestamp(PERIOD_END) + pd.Timedelta(days=1)
parts, total_raw = [], 0
for f in tashu_files:
    part = read_csv_any(f, usecols=use)
    total_raw += len(part)
    part[COL_DATE] = pd.to_datetime(part[COL_DATE], errors='coerce')
    part = part[(part[COL_DATE] >= p_start) & (part[COL_DATE] < p_end)]
    if len(part):
        parts.append(part)
    print(f'  {os.path.basename(f)}: 기간 내 {len(part):,}건')
df = pd.concat(parts, ignore_index=True)
del parts
print(f'원본 전체 {total_raw:,}건 중 분석 기간 {len(df):,}건 사용')

# 좌표 정리
df[COL_LAT] = pd.to_numeric(df[COL_LAT], errors='coerce')
df[COL_LON] = pd.to_numeric(df[COL_LON], errors='coerce')
before = len(df)
df = df.dropna(subset=[COL_DATE, COL_LAT, COL_LON])
df = df[(df[COL_LAT] != 0) & (df[COL_LON] != 0)]
print(f'날짜·좌표 결측 제거: {before - len(df):,}건')

# 위도·경도가 뒤바뀐 경우 자동 교정 (한국 위도는 33~39, 경도는 124~132)
if df[COL_LAT].median() > 100:
    df[[COL_LAT, COL_LON]] = df[[COL_LON, COL_LAT]].values
    print('위도·경도가 바뀌어 있어 교정했습니다.')
lat_med, lon_med = df[COL_LAT].median(), df[COL_LON].median()
print(f'좌표 확인: 위도 중앙값 {lat_med:.4f}, 경도 중앙값 {lon_med:.4f} (대전은 약 36.35, 127.39)')
assert 33 < lat_med < 39 and 124 < lon_med < 132, '좌표가 위경도(도 단위)가 아닙니다. 값 예시를 Claude에게 보여주세요.'

start, end = df[COL_DATE].min().normalize(), df[COL_DATE].max().normalize()
n_days = (end - start).days + 1
print(f'실제 데이터 기간: {start.date()} ~ {end.date()} ({n_days}일)')

agg = {'rentals': (COL_STATION, 'size'),
       'lat': (COL_LAT, 'median'),
       'lon': (COL_LON, 'median'),
       'first_day': (COL_DATE, 'min'),
       'last_day': (COL_DATE, 'max')}
if COL_DIST:
    df[COL_DIST] = pd.to_numeric(df[COL_DIST], errors='coerce')
    agg['avg_dist'] = (COL_DIST, 'mean')

station = df.groupby(COL_STATION).agg(**agg).reset_index().rename(columns={COL_STATION: 'station_id'})
# 대여소별 실제 운영 일수로 나눈 일평균 (기간 중 신설된 대여소의 과소평가 방지)
station['active_days'] = (station['last_day'].dt.normalize() - station['first_day'].dt.normalize()).dt.days + 1
n_short = (station['active_days'] < MIN_ACTIVE_DAYS).sum()
station = station[station['active_days'] >= MIN_ACTIVE_DAYS].copy()
print(f'운영 {MIN_ACTIVE_DAYS}일 미만 대여소 제외: {n_short}개')
station['daily_avg'] = station['rentals'] / station['active_days']
station = station.drop(columns=['first_day', 'last_day']).sort_values('daily_avg', ascending=False)

station.to_csv(os.path.join(PRE, '타슈_대여소별수요.csv'), index=False, encoding='utf-8-sig')
print(f'대여소 수: {len(station):,}개 → 저장 완료')
display(station.head(10))

## 2. 버스정류장 → 대전·청주 추출
공공데이터포털 '국토교통부 전국 버스정류장 위치정보' CSV를 사용한다.

⚠️ 청주와 세종은 붙어 있어서 좌표 범위로 자르면 세종 정류장이 섞인다. 그래서 **도시명 컬럼**으로 거른다.

In [ ]:
bus_file = sorted(glob.glob(os.path.join(RAW, 'bus', '*.csv')))[0]
bus = read_csv_any(bus_file)
print('전체 컬럼:', list(bus.columns))

B_LAT  = find_col(bus.columns, ['위도'])
B_LON  = find_col(bus.columns, ['경도'])
B_CITY = [c for c in bus.columns if any(k in str(c) for k in ['도시', '지자체', '시군']) and '코드' not in str(c)]
print('위도:', B_LAT, '/ 경도:', B_LON, '/ 도시 컬럼:', B_CITY)
assert B_CITY, '도시(지자체) 컬럼을 못 찾았습니다. 전체 컬럼 목록을 보고 B_CITY에 컬럼 이름을 리스트로 직접 지정하세요.'

In [ ]:
def pick_city(df, keyword):
    mask = np.zeros(len(df), dtype=bool)
    for c in B_CITY:
        mask |= df[c].astype(str).str.contains(keyword, na=False)
    out = df[mask].copy()
    out['lat'] = pd.to_numeric(out[B_LAT], errors='coerce')
    out['lon'] = pd.to_numeric(out[B_LON], errors='coerce')
    return out.dropna(subset=['lat', 'lon'])

bus_dj = pick_city(bus, '대전')
bus_cj = pick_city(bus, '청주')
print(f'대전 정류장: {len(bus_dj):,}개 / 청주 정류장: {len(bus_cj):,}개')

bus_dj.to_csv(os.path.join(PRE, '버스정류장_대전.csv'), index=False, encoding='utf-8-sig')
bus_cj.to_csv(os.path.join(PRE, '버스정류장_청주.csv'), index=False, encoding='utf-8-sig')

## 3. 행정동 경계 자동 다운로드 (대전·청주)
공개 행정동 경계 파일(vuski/admdongkor, 2026.7.1 기준)을 받아 대전·청주만 저장한다. 인구·종사자 수를 연결할 때 `adm_cd2`(행정동 코드 10자리)를 기준으로 쓴다.

In [ ]:
import json, urllib.request
URL = 'https://raw.githubusercontent.com/vuski/admdongkor/master/ver20260701/HangJeongDong_ver20260701.geojson'
os.makedirs(os.path.join(RAW, 'boundary'), exist_ok=True)
gj = json.loads(urllib.request.urlopen(URL).read().decode('utf-8'))

def save_city(cond, fname):
    feats = [f for f in gj['features'] if cond(f['properties'])]
    with open(os.path.join(RAW, 'boundary', fname), 'w', encoding='utf-8') as fp:
        json.dump({'type': 'FeatureCollection', 'features': feats}, fp, ensure_ascii=False)
    return pd.DataFrame([f['properties'] for f in feats])

dong_dj = save_city(lambda p: p['sidonm'] == '대전광역시', '행정동경계_대전.geojson')
dong_cj = save_city(lambda p: p['sggnm'].startswith('청주시'), '행정동경계_청주.geojson')
print(f'대전 행정동 {len(dong_dj)}개 / 청주 행정동 {len(dong_cj)}개')
dong_cj[['adm_cd2', 'adm_nm']].to_csv(os.path.join(PRE, '행정동목록_청주.csv'), index=False, encoding='utf-8-sig')
dong_dj[['adm_cd2', 'adm_nm']].to_csv(os.path.join(PRE, '행정동목록_대전.csv'), index=False, encoding='utf-8-sig')
display(dong_cj[['adm_cd2', 'adm_nm']].head(10))

## 4. 시설 좌표 검증 (산단·대학·철도역)
`시설좌표.csv`(Claude가 추정한 좌표, 예전 이름 facilities.csv)를 넣고 실행한다.
1. 지도 검색 서비스(OpenStreetMap)로 이름을 다시 검색해 두 좌표의 거리를 비교한다.
2. 1km 이상 차이 나거나 검색이 안 된 곳은 표로 확인하고, 필요하면 구글 지도에서 좌표를 찾아 FIX에 적는다.

⚠️ 추정 좌표는 그대로 쓰지 말고 반드시 이 검증을 거친다. 특히 '신뢰도 낮음' 항목.

In [ ]:
!pip -q install geopy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

fac_path = os.path.join(RAW, '시설좌표.csv')
if not os.path.exists(fac_path):                      # 예전 이름 호환
    fac_path = os.path.join(RAW, 'facilities.csv')
fac = read_csv_any(fac_path)
geocode = RateLimiter(Nominatim(user_agent='cheongju-public-bike-study').geocode, min_delay_seconds=1.1)

def haversine_m(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2 * 6_371_000 * np.arcsin(np.sqrt(a))

osm_lat, osm_lon = [], []
for _, r in fac.iterrows():
    name = str(r['이름']).split('(')[0].strip()
    city = '청주시' if r['도시'] == '청주' else '대전광역시'
    loc = geocode(f'{name}, {city}', country_codes='kr') or geocode(name, country_codes='kr')
    osm_lat.append(loc.latitude if loc else np.nan)
    osm_lon.append(loc.longitude if loc else np.nan)
fac['검색_위도'], fac['검색_경도'] = osm_lat, osm_lon
fac['차이_m'] = haversine_m(fac['위도'], fac['경도'], fac['검색_위도'], fac['검색_경도']).round(0)
fac['판정'] = np.where(fac['검색_위도'].isna(), '검색 실패 → 지도 확인',
              np.where(fac['차이_m'] > 1000, '차이 큼 → 지도 확인', '일치'))
fac.to_csv(os.path.join(PRE, '시설좌표_검증.csv'), index=False, encoding='utf-8-sig')
print(fac['판정'].value_counts())
display(fac[fac['판정'] != '일치'][['도시', '유형', '이름', '신뢰도', '차이_m', '판정']])

In [ ]:
# ▼ 최종 좌표 확정: 기본은 '일치'면 검색 좌표, 나머지는 추정 좌표를 쓴다.
#   표를 보고 고칠 곳이 있으면 아래 FIX에 직접 적는다. 예: '청주역': (36.xxxx, 127.xxxx)
FIX = {
}
use_osm = fac['판정'] == '일치'
fac['최종_위도'] = np.where(use_osm, fac['검색_위도'], fac['위도'])
fac['최종_경도'] = np.where(use_osm, fac['검색_경도'], fac['경도'])
for name, (la, lo) in FIX.items():
    fac.loc[fac['이름'] == name, ['최종_위도', '최종_경도']] = [la, lo]
fac[['도시', '유형', '이름', '최종_위도', '최종_경도']].rename(columns={'최종_위도': 'lat', '최종_경도': 'lon'})    .to_csv(os.path.join(PRE, '시설좌표_최종.csv'), index=False, encoding='utf-8-sig')
print('저장 완료: 02_전처리/시설좌표_최종.csv')

## 5. 행정동별 종사자 수 (SGIS 원본 → 전처리)

**원본:** 통계청 SGIS 자료제공 → '행정구역별 통계(사업체) – 산업분류별(11차 대분류) 종사자수', 통계년도 2024. 대전(코드 25)·충북(코드 33) 2개 파일을 `01_원본/workers/`에 넣는다. 이미 정리된 파일이 같이 있어도 자동으로 건너뛴다.

| 단계 | 처리 내용 |
|---|---|
| ① 읽기 | 제목 줄 없는 4칸(연도, 지역코드, 업종번호, 종사자수)에 컬럼명을 붙여 읽음 |
| ② 걸러내기 | 지역코드 8자리(읍면동)만 분석에 사용, 2·5자리(시도·시군구)는 검증용 |
| ③ 비밀보호 값 | 가려진 `N/A`의 손실 규모를 상위 지역 합계와 비교해 확인 → 1% 미만이면 0으로 처리 |
| ④ 합치기 | 업종 19개를 행정동별로 합산(전체), 업종번호 003(제조업)은 따로 추출 |
| ⑤ 연결 | 8자리 코드 = 행정동 경계의 `adm_cd`로 연결 (125개 동) |
| ⑥ 저장 | `02_전처리/행정동별_종사자수.csv` |

In [ ]:
# ① 읽기: SGIS 원본은 제목 줄 없이 4칸(연도, 지역코드, 업종번호, 종사자수)이다.
#    코드 앞자리 0이 사라지지 않게 문자(str)로, 'N/A'는 글자 그대로 읽는다.
COLS = ['year', 'code', 'industry', 'value']
w_files = []
for f in sorted(glob.glob(os.path.join(RAW, 'workers', '*.csv'))):
    head = read_csv_any(f, nrows=1, header=None)
    if '종사자수_전체' in head.iloc[0].astype(str).tolist():
        print(f'건너뜀(이미 정리된 파일): {os.path.basename(f)}')
        continue
    w_files.append(f)
print('SGIS 원본 파일:', [os.path.basename(f) for f in w_files])
assert len(w_files) >= 2, '대전(25)·충북(33) 원본 2개가 workers 폴더에 있어야 합니다.'

w_raw = pd.concat([read_csv_any(f, header=None, names=COLS, dtype=str,
                                keep_default_na=False) for f in w_files],
                  ignore_index=True)
w_raw['value'] = pd.to_numeric(w_raw['value'], errors='coerce')  # N/A → 결측
print(f'전체 행 수: {len(w_raw):,}')

# ② 지역코드 자릿수로 단위 구분: 2자리=시도, 5자리=시군구, 8자리=읍면동
w_raw['level'] = w_raw['code'].str.len().map({2: '시도', 5: '시군구', 8: '읍면동'})
w_dong  = w_raw[w_raw['level'] == '읍면동'].copy()
w_upper = w_raw[w_raw['level'] != '읍면동'].copy()
print(w_raw.groupby('level')['code'].nunique().to_string())

# ③ 비밀보호(N/A) 값의 손실 규모 = 1 - (읍면동 합계 / 상위 지역 합계)
print(f"\n가려진 칸(N/A): {w_dong['value'].isna().sum()}개 / {len(w_dong)}칸")
for city, prefix, lv in [('대전', '25', '시도'), ('청주', '3304', '시군구')]:
    d = w_dong[w_dong['code'].str.startswith(prefix)]['value'].sum()
    u = w_upper[(w_upper['level'] == lv) &
                w_upper['code'].str.startswith(prefix)]['value'].sum()
    print(f'{city}: 읍면동 합계 {d:,.0f}명 / 상위 합계 {u:,.0f}명 '
          f'→ 손실 {(1 - d / u) * 100:.2f}%')
print('→ 손실이 1% 미만이므로 N/A는 0으로 보고 합산한다.')

In [ ]:
# ④ 행정동별로 합치기: 전체 종사자 수 + 제조업(업종번호 003) 종사자 수
#    (003 = 11차 표준산업분류 대분류 C 제조업. SGIS 코드표로 최종 확인 권장)
w_dong['is_mfg'] = w_dong['industry'].str.endswith('003')
workers = w_dong.groupby('code').agg(종사자수_전체=('value', 'sum')).reset_index()
mfg = (w_dong[w_dong['is_mfg']].groupby('code')['value'].sum()
       .rename('종사자수_제조업').reset_index())
workers = workers.merge(mfg, on='code', how='left')

# ⑤ 행정동 경계와 연결 (SGIS 8자리 코드 = 경계의 adm_cd)
def load_dong(fname):
    with open(os.path.join(RAW, 'boundary', fname), encoding='utf-8') as fp:
        return pd.DataFrame([f['properties'] for f in json.load(fp)['features']])

dong_all = pd.concat([load_dong('행정동경계_대전.geojson').assign(도시='대전'),
                      load_dong('행정동경계_청주.geojson').assign(도시='청주')],
                     ignore_index=True)

workers_dong = (dong_all[['도시', 'adm_cd', 'adm_cd2', 'adm_nm']]
                .merge(workers, left_on='adm_cd', right_on='code', how='left')
                .drop(columns='code'))
n_ok = workers_dong['종사자수_전체'].notna().sum()
print(f'연결 성공: {n_ok}개 / 경계 {len(dong_all)}개')
cols = ['종사자수_전체', '종사자수_제조업']
workers_dong[cols] = workers_dong[cols].fillna(0).astype(int)

# ⑥ 저장 + 청주 상위 5개 동 확인
workers_dong.to_csv(os.path.join(PRE, '행정동별_종사자수.csv'),
                    index=False, encoding='utf-8-sig')
print('저장 완료: 02_전처리/행정동별_종사자수.csv\n')
top = (workers_dong[workers_dong['도시'] == '청주']
       .nlargest(5, '종사자수_전체').copy())
top['행정동'] = top['adm_nm'].str.split().str[-1]
print(top[['행정동'] + cols].to_string(index=False))

## 6. 행정동별 인구 전처리 (행정안전부 주민등록)

**입력:** jumin.mois.go.kr에서 받은 '행정동별 주민등록 인구 및 세대현황' CSV를 `01_원본/population/`에 넣는다(대전·청주 파일 여러 개 가능).
**출력:** `02_전처리/행정동별_인구.csv`

주민등록 파일은 행정구역 이름 옆 괄호 안에 **10자리 행정기관코드**가 붙어 있다(예: `충청북도 청주시 상당구 중앙동(4311152500)`). 이 코드가 경계 파일의 `adm_cd2`와 같아서 그대로 연결할 수 있다. 숫자에는 쉼표가 들어 있으므로 제거한다.

In [ ]:
import re
p_files = sorted(glob.glob(os.path.join(RAW, 'population', '*.csv')))
pop_raw = pd.concat([read_csv_any(f, dtype=str) for f in p_files], ignore_index=True)
print('컬럼:', list(pop_raw.columns))
display(pop_raw.head(3))

# 행정기관코드 찾기: ① '코드'가 들어간 컬럼이 있으면 사용, ② 없으면 행정구역 이름 속 괄호(10자리)에서 추출
code_col = find_col(pop_raw.columns, ['코드'])
name_col = find_col(pop_raw.columns, ['행정'], exclude=['코드']) or pop_raw.columns[0]
if code_col:
    pop_raw['adm_cd2'] = pop_raw[code_col].str.extract(r'([0-9]{10})')[0]
else:
    pop_raw['adm_cd2'] = pop_raw[name_col].str.extract(r'[(]([0-9]{10})[)]')[0]

# 총인구 컬럼 찾기 (예: '2025년09월_총인구수')
POP_COL = find_col(pop_raw.columns, ['총인구']) or find_col(pop_raw.columns, ['인구'], exclude=['남', '여'])
print('코드 출처:', code_col or name_col, '/ 인구 컬럼:', POP_COL)

pop_raw['인구'] = pd.to_numeric(pop_raw[POP_COL].str.replace(',', ''), errors='coerce')
population = pop_raw.dropna(subset=['adm_cd2', '인구'])[['adm_cd2', '인구']].drop_duplicates('adm_cd2')

# 경계와 연결 점검 (시·구 합계 행은 경계에 없으므로 자연스럽게 빠진다)
pop_dong = dong_all[['도시', 'adm_cd', 'adm_cd2', 'adm_nm']].merge(population, on='adm_cd2', how='left')
miss = pop_dong[pop_dong['인구'].isna()]
print(f'연결 성공: {pop_dong["인구"].notna().sum()}개 / 경계 {len(pop_dong)}개')
if len(miss):
    print('⚠️ 연결 안 된 동(행정동 개편 가능성):', miss['adm_nm'].tolist())
pop_dong.to_csv(os.path.join(PRE, '행정동별_인구.csv'), index=False, encoding='utf-8-sig')

## 7. 변수 계산 함수 → 대전 학습 표

대전 대여소와 청주 격자에 **똑같은 함수**로 변수를 계산한다. 같은 함수를 쓰기 때문에 두 도시의 변수 정의가 정확히 같다는 것이 보장된다(보고서에서 강조할 점).

| 변수 | 계산 방법 |
|---|---|
| bus_cnt | 반경 RADIUS_M 안의 버스정류장 수 |
| pop_density | 지점이 속한 행정동의 인구 ÷ 면적(km²) |
| worker_density | 지점이 속한 행정동의 종사자 수 ÷ 면적(km²) |
| mfg_density | 지점이 속한 행정동의 제조업 종사자 수 ÷ 면적(km²) |
| dist_univ_km / dist_rail_km / dist_ind_km | 가장 가까운 대학·철도역·산업단지까지 직선거리(km) |

⚠️ 인구·종사자는 행정동 단위 평균 밀도라서, 면적이 넓은 면 지역은 산지와 마을이 섞여 밀도가 낮게 나온다. 보고서 '한계'에 적는다.

In [ ]:
!pip -q install geopandas
import geopandas as gpd
from sklearn.neighbors import BallTree

RADIUS_M = 500        # 대여소·격자 주변 버스정류장을 셀 반경(m)
EARTH_R  = 6_371_000  # 지구 반지름(m)

# 행정동 경계 + 인구 + 종사자 → 밀도 계산 (면적은 미터 단위 좌표계 EPSG:5179로 계산)
dong_gdf = pd.concat([gpd.read_file(os.path.join(RAW, 'boundary', '행정동경계_대전.geojson')),
                      gpd.read_file(os.path.join(RAW, 'boundary', '행정동경계_청주.geojson'))], ignore_index=True)
dong_gdf['area_km2'] = dong_gdf.to_crs(5179).area / 1e6
dong_gdf = (dong_gdf.merge(pop_dong[['adm_cd', '인구']], on='adm_cd', how='left')
                    .merge(workers_dong[['adm_cd', '종사자수_전체', '종사자수_제조업']], on='adm_cd', how='left'))
dong_gdf['pop_density']    = dong_gdf['인구'] / dong_gdf['area_km2']
dong_gdf['worker_density'] = dong_gdf['종사자수_전체'] / dong_gdf['area_km2']
dong_gdf['mfg_density']    = dong_gdf['종사자수_제조업'] / dong_gdf['area_km2']

facilities = read_csv_any(os.path.join(PRE, '시설좌표_최종.csv'))
BUS = {'대전': bus_dj, '청주': bus_cj}

def nearest_km(points, targets):
    # 각 지점에서 가장 가까운 대상까지의 직선거리(km)
    tree = BallTree(np.radians(targets[['lat', 'lon']].values), metric='haversine')
    d, _ = tree.query(np.radians(points[['lat', 'lon']].values), k=1)
    return d[:, 0] * EARTH_R / 1000

def compute_features(points, city, radius_m=RADIUS_M):
    # points: lat, lon 컬럼이 있는 표 / city: '대전' 또는 '청주' / 대여소·격자 모두 이 함수로 계산한다
    pts = points.copy()
    # (1) 반경 내 버스정류장 수
    bt = BallTree(np.radians(BUS[city][['lat', 'lon']].values), metric='haversine')
    pts['bus_cnt'] = bt.query_radius(np.radians(pts[['lat', 'lon']].values), r=radius_m / EARTH_R, count_only=True)
    # (2) 지점이 속한 행정동의 밀도 값 붙이기 (점-면 공간 결합)
    g = gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts['lon'], pts['lat']), crs=4326)
    g = gpd.sjoin(g, dong_gdf[['adm_cd', 'adm_nm', 'pop_density', 'worker_density', 'mfg_density', 'geometry']],
                  how='left', predicate='within').drop(columns=['geometry', 'index_right'])
    pts = pd.DataFrame(g)
    # (3) 가장 가까운 시설까지 거리
    fac_c = facilities[facilities['도시'] == city]
    for typ, col in [('대학', 'dist_univ_km'), ('철도역', 'dist_rail_km'), ('산업단지', 'dist_ind_km')]:
        pts[col] = nearest_km(pts, fac_c[fac_c['유형'] == typ])
    return pts

FEATURES = ['bus_cnt', 'pop_density', 'worker_density', 'mfg_density', 'dist_univ_km', 'dist_rail_km', 'dist_ind_km']

train = compute_features(station[['station_id', 'lat', 'lon', 'daily_avg'] + (['avg_dist'] if 'avg_dist' in station else [])], '대전')
out_of_daejeon = train['adm_cd'].isna().sum()
print(f'대전 행정동 밖에 찍힌 대여소: {out_of_daejeon}개 (0이 아니면 좌표 확인)')
train = train.dropna(subset=['adm_cd'])
train.to_csv(os.path.join(PRE, '학습데이터_대전.csv'), index=False, encoding='utf-8-sig')
print(f'학습 표 저장: {len(train)}개 대여소 × 변수 {len(FEATURES)}개')
display(train[FEATURES + ['daily_avg']].describe().round(2))

## 8. 청주 500m 격자 생성 → 예측용 표

청주 행정동 경계 안에 500m 간격 격자 중심점을 만들고, 7번과 같은 함수로 변수를 계산한다. 인구와 종사자가 모두 거의 없는 격자(산지·농경지)는 `거주근무_없음` 표시만 해 두고, 제외 기준은 모델링 단계에서 확정한다.

In [ ]:
GRID_M = 500   # ← 회의 결정에 따라 변경

cj_shape = dong_gdf[dong_gdf['adm_cd'].str.startswith('3304')].to_crs(5179)
cj_union = cj_shape.union_all() if hasattr(cj_shape, 'union_all') else cj_shape.unary_union
minx, miny, maxx, maxy = cj_union.bounds
xs = np.arange(minx + GRID_M / 2, maxx, GRID_M)
ys = np.arange(miny + GRID_M / 2, maxy, GRID_M)
xx, yy = np.meshgrid(xs, ys)
grid = gpd.GeoDataFrame(geometry=gpd.points_from_xy(xx.ravel(), yy.ravel()), crs=5179)
grid = grid[grid.within(cj_union)].to_crs(4326)
grid = pd.DataFrame({'grid_id': range(len(grid)), 'lat': grid.geometry.y.values, 'lon': grid.geometry.x.values})
print(f'청주 격자 수: {len(grid):,}개 ({GRID_M}m 간격)')

pred = compute_features(grid, '청주')
pred['거주근무_없음'] = (pred['pop_density'].fillna(0) < 10) & (pred['worker_density'].fillna(0) < 10)  # km²당 10명 미만
print(f'거주·근무 거의 없음 표시: {pred["거주근무_없음"].sum():,}개 격자')
pred.to_csv(os.path.join(PRE, '예측데이터_청주격자.csv'), index=False, encoding='utf-8-sig')
print('예측용 표 저장: 02_전처리/예측데이터_청주격자.csv')

# 두 도시 변수 범위 비교: 청주 값이 대전 학습 범위를 크게 벗어나면 예측 신뢰도가 낮아진다
cmp = pd.DataFrame({'대전_대여소_중앙값': train[FEATURES].median(), '청주_격자_중앙값': pred[FEATURES].median()}).round(2)
display(cmp)

## 9. 파일 이름 한국어로 정리

예전 버전 노트북으로 만든 영어 이름 파일(`train_daejeon.csv` 등)을 한국어 이름으로 바꾼다. **전처리를 다시 돌리지 않고 이름만 바꾼다.** 이 노트북으로 처음부터 실행했다면 이미 한국어 이름이라 아무 일도 일어나지 않는다.

| 예전 이름 | 새 이름 |
|---|---|
| tashu_station_demand.csv | 타슈_대여소별수요.csv |
| bus_daejeon / bus_cheongju.csv | 버스정류장_대전 / 청주.csv |
| dong_list_*.csv | 행정동목록_대전 / 청주.csv |
| facilities_check / final.csv | 시설좌표_검증 / 최종.csv |
| workers_by_dong.csv | 행정동별_종사자수.csv |
| population_by_dong.csv | 행정동별_인구.csv |
| **train_daejeon.csv** | **학습데이터_대전.csv** |
| **predict_cheongju_grid.csv** | **예측데이터_청주격자.csv** |
| dong_*.geojson | 행정동경계_대전 / 청주.geojson |
| facilities.csv | 시설좌표.csv |
| tashu_00.csv … | 타슈_월별_00.csv … |

In [ ]:
# ▼ 이미 만들어진 파일 이름을 한국어로 바꾼다 (전처리를 다시 돌릴 필요 없음, 여러 번 실행해도 안전)
import os, glob

RENAME = {
    # 02_전처리 결과
    (PRE, 'tashu_station_demand.csv'):   '타슈_대여소별수요.csv',
    (PRE, 'bus_daejeon.csv'):            '버스정류장_대전.csv',
    (PRE, 'bus_cheongju.csv'):           '버스정류장_청주.csv',
    (PRE, 'dong_list_daejeon.csv'):      '행정동목록_대전.csv',
    (PRE, 'dong_list_cheongju.csv'):     '행정동목록_청주.csv',
    (PRE, 'facilities_check.csv'):       '시설좌표_검증.csv',
    (PRE, 'facilities_final.csv'):       '시설좌표_최종.csv',
    (PRE, 'workers_by_dong.csv'):        '행정동별_종사자수.csv',
    (PRE, 'population_by_dong.csv'):     '행정동별_인구.csv',
    (PRE, 'train_daejeon.csv'):          '학습데이터_대전.csv',
    (PRE, 'predict_cheongju_grid.csv'):  '예측데이터_청주격자.csv',
    (PRE, 'tashu_station_with_bus.csv'): '타슈_대여소별수요_정류장포함.csv',
    # 01_원본 안에서 노트북이 만든 파일
    (os.path.join(RAW, 'boundary'), 'dong_daejeon.geojson'):  '행정동경계_대전.geojson',
    (os.path.join(RAW, 'boundary'), 'dong_cheongju.geojson'): '행정동경계_청주.geojson',
    (RAW, 'facilities.csv'):                                  '시설좌표.csv',
}
for (folder, old), new in RENAME.items():
    src, dst = os.path.join(folder, old), os.path.join(folder, new)
    if os.path.exists(src) and not os.path.exists(dst):
        os.rename(src, dst)
        print(f'✅ {old}  →  {new}')

# zip에서 풀린 타슈 월별 파일: tashu_00.csv → 타슈_월별_00.csv
for src in sorted(glob.glob(os.path.join(RAW, 'tashu', 'tashu_*.csv'))):
    num = os.path.basename(src)[len('tashu_'):]
    dst = os.path.join(RAW, 'tashu', '타슈_월별_' + num)
    if not os.path.exists(dst):
        os.rename(src, dst)
        print(f'✅ {os.path.basename(src)}  →  {os.path.basename(dst)}')

print('\n이름 정리 완료. 현재 02_전처리 파일:')
for f in sorted(os.listdir(PRE)):
    print('  -', f)